In [1]:
import os
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_PATH = '/content/drive/MyDrive/sales.csv'
    OUT_DIR = '/content/drive/MyDrive'
except ImportError:
    DATA_PATH = 'sales.csv'
    OUT_DIR = 'results'
os.makedirs(OUT_DIR, exist_ok=True)

Mounted at /content/drive


In [2]:
def make_demo_sales(n=3636, seed=42):
    rng = np.random.default_rng(seed)
    prices = {'Latte': 38.7, 'Hot Chocolate': 38.7, 'Americano': 28.9,
              'Americano with Milk': 33.8, 'Cappuccino': 38.7, 'Cocoa': 35.76,
              'Cortado': 30.0, 'Espresso': 25.96}
    names = rng.choice(list(prices), size=n)
    start, end = pd.Timestamp('2024-03-01 10:00'), pd.Timestamp('2025-03-23 19:00')
    dt = pd.Series(pd.to_datetime(np.sort(rng.integers(start.value, end.value, size=n))))
    cash_type = np.where(rng.random(n) < 0.9, 'card', 'cash')
    card = [f'ANON-0000-0000-{i:04d}' if t == 'card' else None
            for t, i in zip(cash_type, rng.integers(1, 1200, size=n))]
    return pd.DataFrame({
        'date': dt.dt.strftime('%Y-%m-%d'),
        'datetime': dt.dt.strftime('%Y-%m-%d %H:%M:%S.%f').str[:-3],
        'cash_type': cash_type,
        'card': card,
        'money': [prices[x] for x in names],
        'coffee_name': names,
    })

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
else:
    print(f'Файл {DATA_PATH} не знайдено - використовую демонстраційні дані.')
    df = make_demo_sales()

df.head()

,date,datetime,cash_type,card,money,coffee_name
0,2024-03-01,2024-03-01 11:58:48.618000,card,ANON-0000-0000-0450,33.80,Hot chocolate
1,2024-03-01,2024-03-01 13:43:22.828000,card,ANON-0000-0000-0207,28.90,Americano
2,2024-03-01,2024-03-01 14:33:30.639000,cash,NaN,35.76,Cocoa
3,2024-03-01,2024-03-01 14:52:46.593000,card,ANON-0000-0000-0484,33.80,Americano with milk
4,2024-03-01,2024-03-01 15:16:40.466000,card,ANON-0000-0000-0285,38.70,Latte


Колонка `datetime` поки має текстовий тип (`object`) — перетворимо її на кроці 5.

In [3]:
data_messy = {
    'name': ['Anna', None, 'Bohdan', 'Olena', None],
    'age': [20, 21, None, 22, 24],
    'score': [85.5, 90.0, 78.3, None, 92.1]
}
df_messy = pd.DataFrame(data_messy)
df_messy

,name,age,score
0,Anna,20.0,85.5
1,None,21.0,90.0
2,Bohdan,NaN,78.3
3,Olena,22.0,NaN
4,None,24.0,92.1


In [4]:
print(df_messy.isnull().sum())

name     2
age      1
score    1
dtype: int64


У колонці `name` два пропуски, в `age` та `score` — по одному.

In [5]:
df_dropped = df_messy.dropna(subset=['name'])
print(df_dropped)

mean_age = df_messy['age'].mean()
df_filled = df_messy.copy()
df_filled['age'] = df_filled['age'].fillna(mean_age)
print(df_filled)

     name   age  score
0    Anna  20.0   85.5
2  Bohdan   NaN   78.3
3   Olena  22.0    NaN
     name    age  score
0    Anna  20.00   85.5
1    None  21.00   90.0
2  Bohdan  21.75   78.3
3   Olena  22.00    NaN
4    None  24.00   92.1


## Крок 3: Виявлення та видалення дублікатів

In [6]:
data_dup = {
    'id': [1, 2, 3, 2, 3],
    'value': ['A', 'B', 'C', 'B', 'C']
}
df_dup = pd.DataFrame(data_dup)
df_dup

,id,value
0,1,A
1,2,B
2,3,C
3,2,B
4,3,C


In [7]:
print(df_dup.duplicated())
print("Кількість дублікатів:", df_dup.duplicated().sum())

0    False
1    False
2    False
3     True
4     True
dtype: bool
Кількість дублікатів: 2


In [8]:
df_unique = df_dup.drop_duplicates()
print(df_unique)

   id value
0   1     A
1   2     B
2   3     C


In [9]:
Q1 = df['money'].quantile(0.25)
Q3 = df['money'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f"Q1 = {Q1}, Q3 = {Q3}, IQR = {IQR}")
print(f"Нижня межа: {lower}, Верхня межа: {upper}")

Q1 = 28.9, Q3 = 38.7, IQR = 9.800000000000004
Нижня межа: 14.199999999999992, Верхня межа: 53.400000000000006


In [10]:
outliers = df[(df['money'] < lower) | (df['money'] > upper)]
print(f"Знайдено викидів: {len(outliers)}")

Знайдено викидів: 0


Якби викиди були, їх можна було б видалити або обмежити граничними значеннями, наприклад `df['money'].clip(lower, upper)`.

In [11]:
print(df.dtypes, "\n")
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
print(df.dtypes)

date            object
datetime        object
cash_type       object
card            object
money          float64
coffee_name     object
dtype: object 

date                   object
datetime       datetime64[ns]
cash_type              object
card                   object
money                 float64
coffee_name            object
dtype: object


In [12]:
df_text = pd.DataFrame({
    'city': ['  Kyiv  ', 'LVIV', 'Odesa ', 'KHARKIV', ' kyiv '],
    'phone': ['(044) 123-45-67', '(032) 234-56-78', '(048) 345-67-89',
              '(057) 456-78-90', '(044) 123-45-67']
})
df_text

,city,phone
0,Kyiv,(044) 123-45-67
1,LVIV,(032) 234-56-78
2,Odesa,(048) 345-67-89
3,KHARKIV,(057) 456-78-90
4,kyiv,(044) 123-45-67


In [13]:
df_text['city'] = df_text['city'].str.strip().str.lower()
df_text['phone'] = df_text['phone'].str.replace(r'\(|\)|-', '', regex=True)
print(df_text)

      city        phone
0     kyiv  044 1234567
1     lviv  032 2345678
2    odesa  048 3456789
3  kharkiv  057 4567890
4     kyiv  044 1234567


In [14]:
df['date_only'] = df['datetime'].dt.date
df['hour'] = df['datetime'].dt.hour
df['day_of_week'] = df['datetime'].dt.dayofweek
df['month'] = df['datetime'].dt.month
print(df[['datetime', 'date_only', 'hour', 'day_of_week', 'month']].head())

                 datetime   date_only  hour  day_of_week  month
0 2024-03-01 11:58:48.618  2024-03-01    11            4      3
1 2024-03-01 13:43:22.828  2024-03-01    13            4      3
2 2024-03-01 14:33:30.639  2024-03-01    14            4      3
3 2024-03-01 14:52:46.593  2024-03-01    14            4      3
4 2024-03-01 15:16:40.466  2024-03-01    15            4      3


In [15]:
df_encoded = pd.get_dummies(df, columns=['coffee_name'])
print(df_encoded.head())

         date                datetime cash_type                 card  money  \
0  2024-03-01 2024-03-01 11:58:48.618      card  ANON-0000-0000-0450  33.80   
1  2024-03-01 2024-03-01 13:43:22.828      card  ANON-0000-0000-0207  28.90   
2  2024-03-01 2024-03-01 14:33:30.639      cash                  NaN  35.76   
3  2024-03-01 2024-03-01 14:52:46.593      card  ANON-0000-0000-0484  33.80   
4  2024-03-01 2024-03-01 15:16:40.466      card  ANON-0000-0000-0285  38.70   

    date_only  hour  day_of_week  month  coffee_name_Americano  \
0  2024-03-01    11            4      3                  False   
1  2024-03-01    13            4      3                   True   
2  2024-03-01    14            4      3                  False   
3  2024-03-01    14            4      3                  False   
4  2024-03-01    15            4      3                  False   

   coffee_name_Americano with milk  coffee_name_Cappuccino  coffee_name_Cocoa  \
0                            False             

Замість однієї текстової колонки `coffee_name` з’явилось кілька бінарних; `True` (1) означає, що рядок належить до відповідної категорії.

In [16]:
df['money_norm'] = (df['money'] - df['money'].min()) / (df['money'].max() - df['money'].min())
df['money_std'] = (df['money'] - df['money'].mean()) / df['money'].std()

In [17]:
print(df[['money', 'money_norm', 'money_std']].head())
print("Мінімум money_norm:", df['money_norm'].min())
print("Максимум money_norm:", df['money_norm'].max())
print("Середнє money_std:", df['money_std'].mean())
print("Стандартне відхилення money_std:", df['money_std'].std())

   money  money_norm  money_std
0  33.80    0.666667  -0.038815
1  28.90    0.333333  -1.065107
2  35.76    0.800000   0.371702
3  33.80    0.666667  -0.038815
4  38.70    1.000000   0.987477
Мінімум money_norm: 0.0
Максимум money_norm: 1.0
Середнє money_std: 1.2292389328649732e-15
Стандартне відхилення money_std: 1.000000000000002


Середнє `money_std` — дуже мале число (порядку 1e-17), фактично нуль. Це наслідок обмеженої точності обчислень із дробовими числами.

In [18]:
final_df = df[['datetime', 'cash_type', 'money', 'coffee_name',
               'hour', 'day_of_week', 'month', 'money_norm', 'money_std']]
final_df.to_csv(os.path.join(OUT_DIR, 'cleaned_sales.csv'), index=False)
print('Збережено:', os.path.join(OUT_DIR, 'cleaned_sales.csv'), final_df.shape)

Збережено: /content/drive/MyDrive/cleaned_sales.csv (1000, 9)


In [19]:
final_df.head()

,datetime,cash_type,money,coffee_name,hour,day_of_week,month,money_norm,money_std
0,2024-03-01 11:58:48.618,card,33.80,Hot chocolate,11,4,3,0.666667,-0.038815
1,2024-03-01 13:43:22.828,card,28.90,Americano,13,4,3,0.333333,-1.065107
2,2024-03-01 14:33:30.639,cash,35.76,Cocoa,14,4,3,0.800000,0.371702
3,2024-03-01 14:52:46.593,card,33.80,Americano with milk,14,4,3,0.666667,-0.038815
4,2024-03-01 15:16:40.466,card,38.70,Latte,15,4,3,1.000000,0.987477


## Висновки
У роботі виконано повний цикл очищення: виявлено й оброблено пропуски, дублікати та викиди, виправлено типи даних, очищено текст, створено часові ознаки, закодовано категорії, масштабовано числові значення та збережено результат у CSV. Усі кроки зафіксовані в коді, тож послідовність відтворювана.